In [ ]:
import os
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

class Dataset_stock(Dataset):
    def __init__(self, data_path, window_size, window_stride, flag='train',
                 feature='S', target='OT', scale=True, timeenc=0,
                 seasonal_patterns=None):
        assert flag in ['train', 'test', 'val']
        type_map = {'train': 0, 'val': 1, 'test': 2}
        self.stride = window_stride
        self.set_type = type_map[flag]
        self.window_size = window_size
        self.feature = feature
        self.target = target
        self.scale = scale
        self.timeenc = timeenc
        self.data_path = data_path
        self.__read_data__()

    def __read_data__(self):
        self.scaler = StandardScaler()
        self.data = [] 
        self.time_stamp = []  
        self.scaler = StandardScaler()
        for data_path in tqdm(self.data_path):
            df_raw = pd.read_csv(data_path)
            num_train = int(len(df_raw) * 0.7)
            num_test = int(len(df_raw) * 0.2)
            num_vali = len(df_raw) - num_train - num_test
            border1s = [0, num_train - self.seq_len, len(df_raw) - num_test - self.seq_len]
            border2s = [num_train, num_train + num_vali, len(df_raw)]
            
            border1 = border1s[self.set_type]
            border2 = border2s[self.set_type]
            
            df_raw = df_raw[['Date',self.feature, self.target]]
            df_data = df_raw[[self.feature, self.target]]
            if self.scale:
                train_data = df_data[border1s[0]:border2s[0]]
                self.scaler.fit(train_data[self.feature].values)
                df_data[self.feature] = self.scaler.transform(df_data[self.feature].values)
                data = df_data
            else:
                data = df_data
        
        # df_stamp = df_raw[['Date']][border1:border2]  # DataFrame
        # df_stamp['Date'] = pd.to_datetime(df_stamp['Date'])  # 修复：直接用 'Date' 列
            data = data.values
            
            # tot_len = (len(data[border1:border2]) - self.window_size) // self.stride + 1
            # if tot_len <= 0:
            #     raise ValueError(f"File {data_path} too short for window_size={self.window_size}, stride={self.stride}")
            
            self.data.append(data[border1:border2])
            # self.time_stamp.append(df_raw['Date'][border1:border2])

        concatenated_data = np.concatenate(all_data, axis=0)
        # concatenated_stamps = pd.concat(all_stamps, ignore_index=True)
            
        #     self.tot_len_per_file.append(tot_len)
        # self.tot_len = tot_len
        # self.total_samples = sum(self.tot_len_per_file)
    def __getitem__(self, index):
        
        # 用除法定位文件和局部索引
        file_idx = index // self.tot_len  # 文件索引
        local_index = index % self.tot_len  # 文件内局部索引
    
        # 按步幅计算窗口起始位置
        s_begin = local_index * self.stride
        s_end = s_begin + self.window_size
        
        # 确保不越界（可选检查）
        if s_end > len(self.data[file_idx]):
            raise ValueError(f"Window end {s_end} exceeds data length {len(self.data[file_idx])}")

        # 提取窗口数据
        window_data = self.data[file_idx][s_begin:s_end]  # (window_size, 2)
        seq_x = window_data[:, 0:1]  # (window_size, 1)，特征序列
        seq_y = window_data[-1, 1:2]  # (1, 1)，窗口最后一个时间步的分类标签
        file_name = os.path.splitext(os.path.basename(self.data_path[file_idx]))[0]
        seq_x = torch.tensor(seq_x, dtype=torch.float32)
        seq_y = torch.tensor(seq_y, dtype=torch.float32)
        return seq_x, seq_y, file_name

    def __len__(self):
        return self.total_samples

